# Домашнее задание. Тема 9. Случайные события. Случайные величины

**Студент:** Кузнецов Данила, группа УЦП-25

Смоделируем игру против лотерейного автомата типа «777»: игрок платит 1 руб., выпадает случайное целое число от 0 до 999 (равномерно).

**Таблица выигрышей** (`*` — любая цифра):

| Комбинация | Выигрыш, руб. |
|---|---|
| 777 | 200 |
| 999 | 100 |
| 555 | 50 |
| 333 | 15 |
| 111 | 10 |
| *77 | 5 |
| **7 | 3 |
| *00 | 2 |
| **0 | 1 |

## Задание 1. Моделирование игры (Монте-Карло)

Проверка комбинаций идёт от самой ценной к менее ценной (777 → 999/555/333/111 → *77 → **7 → *00 → **0), первая подошедшая определяет выигрыш. Число форматируется в три цифры с ведущими нулями (`'007'`). Средний результат считается по 1 000 000 симуляций.

In [1]:
import random


WIN_777 = 200       # 777
WIN_TRIPLE = {'999': 100, '555': 50, '333': 15, '111': 10}
WIN_LAST_TWO = 5    # *77
WIN_LAST_ONE = 3    # **7
WIN_ZERO_TWO = 2    # *00
WIN_ZERO_ONE = 1    # **0
COST = 1            # цена игры


def win_amount(number):
    """Выигрыш по выпавшему числу 0..999 (таблица из условия)."""
    digits = f'{number:03d}'            # '000'..'999' с ведущими нулями
    if digits == '777':
        return WIN_777
    if digits in WIN_TRIPLE:
        return WIN_TRIPLE[digits]
    if digits[1:] == '77':              # *77
        return WIN_LAST_TWO
    if digits[-1] == '7':               # **7
        return WIN_LAST_ONE
    if digits[1:] == '00':              # *00
        return WIN_ZERO_TWO
    if digits[-1] == '0':               # **0
        return WIN_ZERO_ONE
    return 0


def monte_carlo(n_games, seed=42):
    """Симуляция n_games игр: средний результат игрока за одну игру."""
    rng = random.Random(seed)
    total = 0
    for _ in range(n_games):
        total += win_amount(rng.randint(0, 999))
    return (total - n_games * COST) / n_games


n_games = 1_000_000
mc = monte_carlo(n_games)
print(f'Монте-Карло: {n_games:,} игр'.replace(',', ' '))
print(f'Средний результат игрока за игру: {mc:.4f} руб.')
if mc < 0:
    print(f'Игра невыгодна: игрок в среднем теряет {abs(mc) * 100:.0f} коп. за игру.')

Монте-Карло: 1000000 игр
Средний результат игрока за игру: -0.1984 руб.
Игра невыгодна: игрок в среднем теряет 20 коп. за игру.


## Задание 2 (дополнительное). Теоретический расчёт

Все 1000 исходов равновероятны (вероятность каждого 1/1000), поэтому точное математическое ожидание выигрыша считается прямым перебором всех исходов:

- 777: 1/1000 × 200 = 0.2 руб.
- 999, 555, 333, 111: 4/1000 × (100+50+15+10)/4 = 0.175 руб.
- *77: 9/1000 × 5 = 0.045 руб. (077, 177, ..., 977 — без 777)
- **7: 90/1000 × 3 = 0.27 руб. (все *XY7, кроме 777 и *77)
- *00: 10/1000 × 2 = 0.02 руб. (000, 100, ..., 900)
- **0: 90/1000 × 1 = 0.09 руб. (все *XY0, кроме *00)

Итого матожидание выигрыша 0.80 руб. при стоимости игры 1 руб.

In [2]:
def theoretical_ev():
    """Точный средний выигрыш за игру: перебор всех 1000 равновероятных исходов."""
    ev = sum(win_amount(number) for number in range(1000)) / 1000
    return ev - COST


ev = theoretical_ev()
print(f'Теоретический средний результат за игру: {ev:.4f} руб.')
print(f'Расхождение модели и теории: {abs(mc - ev):.4f} руб.')

Теоретический средний результат за игру: -0.2000 руб.
Расхождение модели и теории: 0.0016 руб.


## Вывод

Игра **невыгодна** игроку: теоретический средний результат —0.2 руб. за игру (матожидание выигрыша 0.80 руб. при цене игры 1 руб.). Монте-Карло на 1 000 000 игр даёт −0.1984 руб. — расхождение с теорией всего 0.0016 руб., что подтверждает корректность модели. Именно такая маржа и обеспечивает автомату прибыль.